# Capa Silver — Limpieza, estandarización y modelo dimensional
**Proyecto:** Detección de Fraude en Transacciones con Tarjeta de Crédito
**Objetivo de la capa:** A partir de la tabla Bronze, aplicar tratamiento de nulos, conversión de tipos de datos, estandarización de nombres de columnas, eliminación de duplicados, creación de columnas calculadas y validación de reglas de negocio; y construir el modelo dimensional (dimensiones + tabla de hechos) que servirá de insumo a la capa Gold.

In [0]:
catalog = "fraud_proyecto"
schema_name = "silver"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_name}")

df_original = spark.table(f"{catalog}.bronze.fraud_test_original")
print(f"Registros leídos desde Bronze: {df_original.count():,}")

## 1. Tratamiento de nulos

Se identifican nulos en los campos críticos del análisis (identificador de transacción, monto, fecha y bandera de fraude) y se descartan los registros incompletos en esos campos, dejando evidencia del volumen depurado.

In [0]:
from pyspark.sql.functions import col

campos_criticos = ["trans_num", "cc_num", "amt", "trans_date_trans_time", "is_fraud"]

condicion_nulos = None
for c in campos_criticos:
    cond = col(c).isNull()
    condicion_nulos = cond if condicion_nulos is None else (condicion_nulos | cond)

registros_con_nulos = df_original.filter(condicion_nulos).count()
print(f"Registros con nulos en campos críticos {campos_criticos}: {registros_con_nulos}")

df_sin_nulos_criticos = df_original.filter(~condicion_nulos)
print(f"Registros tras descartar nulos críticos: {df_sin_nulos_criticos.count():,}")

## 2. Estandarización de texto y nombres de columnas

Se homogeniza el formato de los campos de texto (minúsculas, sin espacios) y se renombran las columnas originales a un estándar de negocio en español, cumpliendo el requisito de *estandarización de nombres de columnas*.

In [0]:
from pyspark.sql.functions import trim, lower
#Limpieza de texto sobre las columnas originales
df_texto_limpio = (
    df_sin_nulos_criticos
    .withColumn("merchant", trim(lower(col("merchant"))))
    .withColumn("category", trim(lower(col("category"))))
    .withColumn("first", trim(col("first")))
    .withColumn("last", trim(col("last")))
    .withColumn("job", trim(col("job")))
    .withColumn("street", trim(col("street")))
    .withColumn("city", trim(col("city")))
    .withColumn("state", trim(col("state")))
)

#Estandarización de nombres de columnas (snake_case, español de negocio)
mapa_renombrado = {
    "trans_date_trans_time": "fecha_hora_transaccion_raw",
    "cc_num":                "numero_tarjeta",
    "merchant":               "nombre_comercio",
    "category":               "nombre_categoria",
    "amt":                    "monto",
    "first":                  "nombre_titular",
    "last":                   "apellido_titular",
    "gender":                 "genero_titular",
    "street":                 "direccion_titular",
    "city":                   "ciudad_titular",
    "state":                  "estado_titular",
    "zip":                    "codigo_postal_titular",
    "lat":                    "latitud_titular",
    "long":                   "longitud_titular",
    "city_pop":               "poblacion_ciudad",
    "job":                    "ocupacion_titular",
    "dob":                    "fecha_nacimiento_raw",
    "trans_num":              "id_transaccion",
    "unix_time":              "unix_time",
    "merch_lat":              "latitud_comercio",
    "merch_long":             "longitud_comercio",
    "is_fraud":               "es_fraude_raw",
}

df_renombrado = df_texto_limpio
for nombre_original, nombre_nuevo in mapa_renombrado.items():
    df_renombrado = df_renombrado.withColumnRenamed(nombre_original, nombre_nuevo)

df_renombrado.printSchema()

## 3. Conversión de tipos de datos

> **Corrección aplicada:** el campo `trans_date_trans_time` llega con el formato `dd-MM-yyyy HH:mm` (con **guiones**, ej. `21-06-2020 12:14`). En una versión anterior se usó por error el patrón `dd/MM/yyyy HH:mm` (con barras), lo que provocaba que `try_to_timestamp` devolviera `NULL` en el 100% de los registros y, en consecuencia, que la regla de negocio de fecha válida (sección 4) rechazara **todas** las filas. Se corrige el patrón de formato a guiones y se deriva `fecha_transaccion` directamente desde el timestamp ya parseado (en vez de volver a parsear un substring), para evitar una segunda fuente de desincronización de formato.
> Adicionalmente, `fecha_nacimiento_raw` ya llega tipado como `date` desde Bronze (Spark lo infirió automáticamente), por lo que se castea directamente en vez de volver a parsear con `try_to_date`.

In [0]:
from pyspark.sql.functions import expr, to_date

df_tipado = (
    df_renombrado
    .withColumn("numero_tarjeta", col("numero_tarjeta").cast("string"))
    .withColumn("monto", col("monto").cast("double"))
    .withColumn("latitud_titular", col("latitud_titular").cast("double"))
    .withColumn("longitud_titular", col("longitud_titular").cast("double"))
    .withColumn("poblacion_ciudad", col("poblacion_ciudad").cast("long"))
    .withColumn("latitud_comercio", col("latitud_comercio").cast("double"))
    .withColumn("longitud_comercio", col("longitud_comercio").cast("double"))
    .withColumn("es_fraude", col("es_fraude_raw").cast("int"))
    .withColumn(
        "fecha_hora_transaccion",
        expr("try_to_timestamp(fecha_hora_transaccion_raw, 'dd-MM-yyyy HH:mm')")
    )
    .withColumn(
        "fecha_transaccion",
        to_date(col("fecha_hora_transaccion"))
    )
    .withColumn(
        "fecha_nacimiento",
        col("fecha_nacimiento_raw").cast("date")
    )
)

display(df_tipado.limit(10))

# Verificación de control: si esto imprime un número alto, el parseo de fecha está fallando
fallos_fecha = df_tipado.filter(col("fecha_hora_transaccion").isNull()).count()
print(f"Registros con fecha_hora_transaccion no parseada (debe ser 0 o muy bajo): {fallos_fecha}")

## 4. Eliminación de duplicados a nivel de registro

In [0]:
df_clean = df_validos.dropDuplicates(["id_transaccion"])
print(f"Registros tras eliminar duplicados por id_transaccion: {df_clean.count():,}")

## 5. Construcción del modelo dimensional

> **Nota de calidad de datos (no corregible en código):** la columna `numero_tarjeta` (`cc_num` original) llega en el archivo fuente en notación científica (ej. `2.29116E15`), lo que indica que el archivo CSV ya almacena el número de tarjeta con dígitos significativos truncados — probablemente por una exportación previa desde Excel. Esto implica un riesgo real de colisión entre distintos titulares cuyo número de tarjeta se trunque al mismo valor aproximado. Esta limitación proviene del dato fuente, no del pipeline, y debe documentarse como hallazgo de calidad de datos en el informe (sección 7 / diccionario de datos) en lugar de "corregirse", ya que los dígitos perdidos no son recuperables desde este archivo.

### 5.1 Dimensión de titulares de tarjeta

In [0]:
dim_titulares = df_clean.select(
    "numero_tarjeta",
    "nombre_titular", "apellido_titular", "genero_titular",
    "direccion_titular", "ciudad_titular", "estado_titular", "codigo_postal_titular",
    "latitud_titular", "longitud_titular",
    "poblacion_ciudad", "ocupacion_titular", "fecha_nacimiento"
).withColumnRenamed("numero_tarjeta", "id_titular") \
 .dropDuplicates(["id_titular"])

(
    dim_titulares.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.{schema_name}.dim_titulares_tarjeta")
)

display(dim_titulares.limit(10))

### 5.2 Dimensión de comercios

In [0]:
from pyspark.sql.functions import monotonically_increasing_id
dim_comercios = (
    df_clean
    .select("nombre_comercio")
    .dropDuplicates()
    .withColumn("id_comercio", monotonically_increasing_id() + 1)
    .select("id_comercio", "nombre_comercio")
)
(
    dim_comercios.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.{schema_name}.dim_comercios")
)
display(dim_comercios.limit(10))

### 5.3 Dimensión de fechas

In [0]:
from pyspark.sql.functions import year, month, dayofmonth, date_format
dim_fechas = (
    df_clean
    .select("fecha_transaccion")
    .dropDuplicates()
    .withColumnRenamed("fecha_transaccion", "fecha")
    .withColumn("anio", year("fecha"))
    .withColumn("mes", month("fecha"))
    .withColumn("dia", dayofmonth("fecha"))
    .withColumn("nombre_dia", date_format("fecha", "EEEE"))
)
(
    dim_fechas.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.{schema_name}.dim_fechas")
)
display(dim_fechas.limit(10))

### 5.4 Tabla de hechos de transacciones

Incluye las columnas calculadas requeridas para el análisis de fraude: hora, día de la semana, indicador de madrugada, indicador de hora pico y edad del titular al momento de la transacción.

> Se corrigió el operador lógico faltante entre los dos rangos horarios de `es_hora_pico` (9–12 y 18–21), detectado en la versión anterior del notebook.

In [0]:
from pyspark.sql.functions import hour, dayofweek, when, year as year_fn
fact_transacciones = df_clean.select(
    col("id_transaccion"),
    col("numero_tarjeta").alias("id_titular"),
    col("nombre_comercio"),
    col("nombre_categoria"),
    col("fecha_transaccion").alias("fecha"),
    col("fecha_hora_transaccion").alias("timestamp_transaccion"),
    # Columnas derivadas de fecha/hora
    hour(col("fecha_hora_transaccion")).alias("hora"),
    dayofweek(col("fecha_transaccion")).alias("dia_de_la_semana"),
    # Columna calculada: Madrugada
    when(
        (hour(col("fecha_hora_transaccion")) >= 0) & (hour(col("fecha_hora_transaccion")) <= 6),
        True
    ).otherwise(False).alias("es_madrugada"),
    # Columna calculada: Hora Pico
    when(
        ((hour(col("fecha_hora_transaccion")) >= 9) & (hour(col("fecha_hora_transaccion")) <= 12))
        | ((hour(col("fecha_hora_transaccion")) >= 18) & (hour(col("fecha_hora_transaccion")) <= 21)),
        True
    ).otherwise(False).alias("es_hora_pico"),
    # Columna calculada: Edad del titular
    (year_fn(col("fecha_transaccion")) - year_fn(col("fecha_nacimiento"))).alias("edad_del_titular"),
    col("monto"),
    col("es_fraude"),
    col("latitud_comercio"),
    col("longitud_comercio")
)
(
    fact_transacciones.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{catalog}.{schema_name}.fact_transacciones")
)

display(fact_transacciones.limit(10))

## 6. Verificación general de la capa Silver

In [0]:
for tabla in ["dim_titulares_tarjeta", "dim_comercios", "dim_fechas", "fact_transacciones", "fraude_rechazados"]:
    total = spark.table(f"{catalog}.{schema_name}.{tabla}").count()
    print(f"{catalog}.{schema_name}.{tabla}: {total:,} registros")

## 7. Exportación a CSV para conciliación en Excel y Dashboard

In [0]:
tablas_export = ["dim_titulares_tarjeta", "dim_comercios", "dim_fechas", "fact_transacciones"]

spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema_name}.export_csv")

for tabla in tablas_export:
    nombre_tabla = f"{catalog}.{schema_name}.{tabla}"
    ruta_salida = f"/Volumes/{catalog}/{schema_name}/export_csv/{tabla}"

    print(f"Exportando {tabla} ...")
    df = spark.table(nombre_tabla)
    (
        df.write
        .format("csv")
        .option("header", "true")
        .option("sep", ",")
        .option("encoding", "UTF-8")
        .mode("overwrite")
        .save(ruta_salida)
    )
    print(f"  -> Exportado a: {ruta_salida}")

print("Exportación de tablas Silver completada.")

In [0]:
fact_transacciones.printSchema()